# Notebook to compare two boundary defs to see how things have changed.

In [3]:
# Import Required Libraries
import pandas as pd
import geopandas as gpd

#load centres A and centres B
# centres_a = gpd.read_parquet("../data/output_files/mergeafterfiltthresh_retailcen_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
# centres_b = gpd.read_parquet("../data/output_files/aug2025_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
# centres_a = gpd.read_parquet("../data/output_files/nov2025_v5defuntfilter_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
# centres_a = gpd.read_parquet("../data/output_files/nov2025_v3newretailfilter_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
# centres_a = gpd.read_parquet("../data/output_files/nov2025_v6orderswitch_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
centres_a = gpd.read_parquet("../data/output_files/nov2025_v17check_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
centres_b = gpd.read_parquet("../data/output_files/nov2025check_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
old_centres = gpd.read_parquet("../data/old_retail_centres/old_retail_centers.parquet")


In [4]:
#Print some basic stats about the centres
print(f"Centres A: {centres_a.shape[0]} rows")
print(f"Centres B: {centres_b.shape[0]} rows")
print(f"Old Centres: {old_centres.shape[0]} rows")

#print some comparison stats using the geometry intersections
print(f"Centres A and B intersection: {gpd.overlay(centres_a, centres_b, how='intersection').shape[0]} rows")
print(f"Centres A and Old Centres intersection: {gpd.overlay(centres_a, old_centres, how='intersection').shape[0]} rows")
print(f"Centres B and Old Centres intersection: {gpd.overlay(centres_b, old_centres, how='intersection').shape[0]} rows")

# helper function: count how many in A don't intersect anything in B
def count_not_in(A, B):
    return A[~A.geometry.apply(lambda g: B.intersects(g).any())].shape[0]

print(f"Old Centres not in Centres A: {count_not_in(old_centres, centres_a)} rows")
print(f"Old Centres not in Centres B: {count_not_in(old_centres, centres_b)} rows")

print(f"Centres A not in Old Centres: {count_not_in(centres_a, old_centres)} rows")
print(f"Centres B not in Old Centres: {count_not_in(centres_b, old_centres)} rows")

#centres in A not in B
print(f"Centres A not in Centres B: {count_not_in(centres_a, centres_b)} rows")
print(f"Centres B not in Centres A: {count_not_in(centres_b, centres_a)} rows")


Centres A: 10023 rows
Centres B: 10013 rows
Old Centres: 6423 rows


/tmp/ipykernel_240955/874018236.py:7: UserWarning: `keep_geom_type=True` in overlay resulted in 2095 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  print(f"Centres A and B intersection: {gpd.overlay(centres_a, centres_b, how='intersection').shape[0]} rows")


Centres A and B intersection: 10460 rows
Centres A and Old Centres intersection: 6899 rows
Centres B and Old Centres intersection: 6891 rows
Old Centres not in Centres A: 495 rows
Old Centres not in Centres B: 495 rows
Centres A not in Old Centres: 3955 rows
Centres B not in Old Centres: 3952 rows
Centres A not in Centres B: 0 rows
Centres B not in Centres A: 0 rows


In [ ]:

# --- Old centres not intersecting centres_a ---
joined_a = gpd.sjoin(old_centres, centres_a, how="left", predicate="intersects")
old_centres_not_in_a = joined_a[joined_a.index_right.isna()].drop(columns="index_right")

# --- Old centres not intersecting centres_b ---
joined_b = gpd.sjoin(old_centres, centres_b, how="left", predicate="intersects")
old_centres_not_in_b = joined_b[joined_b.index_right.isna()].drop(columns="index_right")

# --- centres_a not intersecting centres_b ---
joined_a_b = gpd.sjoin(centres_a, centres_b, how="left", predicate="intersects")
centres_a_not_in_b = joined_a_b[joined_a_b.index_right.isna()].drop(columns="index_right")

# --- centres_b not intersecting centres_a ---
joined_b_a = gpd.sjoin(centres_b, centres_a, how="left", predicate="intersects")
centres_b_not_in_a = joined_b_a[joined_b_a.index_right.isna()].drop(columns="index_right")

# Save results
old_centres_not_in_a.to_parquet("../data/mappingoutputs/old_centres_not_in_a.parquet")
old_centres_not_in_b.to_parquet("../data/mappingoutputs/old_centres_not_in_b.parquet")
centres_a_not_in_b.to_parquet("../data/mappingoutputs/centres_a_not_in_b.parquet")
centres_b_not_in_a.to_parquet("../data/mappingoutputs/centres_b_not_in_a.parquet")


: 

In [9]:
#find the rowwise differences between centres_a and centres_b and save to file (not geometries)

# Drop geometry column and find differences based on RC_ID or all columns
centres_a_no_geom = centres_a.drop(columns=['geometry'])
centres_b_no_geom = centres_b.drop(columns=['geometry'])

# Convert 'category_names' column to string for hashability
centres_a_no_geom['category_names'] = centres_a_no_geom['category_names'].apply(str)
centres_b_no_geom['category_names'] = centres_b_no_geom['category_names'].apply(str)

# Concatenate and find rows that appear only once (i.e., different between A and B)
diff_a_b = pd.concat([centres_a_no_geom, centres_b_no_geom]).drop_duplicates(keep=False)

In [10]:
diff_a_b 

,RC_ID,Total_POI_count,# H3 cells,Retail,Dining and Drinking,Leisure & Travel,Community & Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3
0,6059,16,5,7.0,6.0,0.0,3.0,['Grocery Store' 'Bar' 'Pub' 'Pharmacy' 'Socia...,centre,North East Derbyshire,None,None
1,6705,14,11,11.0,2.0,1.0,0.0,['Grocery Store' 'Hardware Store' 'Fish and Ch...,centre,Cornwall,None,None
2,1191,131,71,86.0,33.0,4.0,8.0,['Grocery Store' 'Garden Center' 'Arts and Ent...,centre,North East Derbyshire,None,None
3,6060,16,44,11.0,3.0,2.0,0.0,['Gym' 'Fuel Station' 'Shoe Store' 'Bicycle St...,centre,North East Derbyshire,None,None
4,7508,12,4,6.0,3.0,1.0,2.0,['Dance Studio' 'Hardware Store' 'Italian Rest...,centre,Chesterfield,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
10008,3994,29,22,17.0,8.0,2.0,2.0,['Bakery' 'Butcher' 'Post Office' 'Arts and Cr...,retail_park,Braintree,None,None
10009,4125,28,9,17.0,10.0,0.0,1.0,['Gourmet Store' 'Eyecare Store' 'Post Office'...,retail_park,Haringey,None,None
10010,2646,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Jewelry Store' 'Vape Store'...,retail_park,"Bournemouth, Christchurch and Poole",None,None
10011,3717,32,22,26.0,6.0,0.0,0.0,['Furniture and Home Store' 'Eyecare Store' 'P...,retail_park,Glasgow City,None,None
